# Healthcare Analytics for Doctor Visits
## Objective
Analyze the supplied healthcare dataset to identify patterns associated with doctor visits. The supplied guide describes 5,190 records and 13 columns. This is an educational analysis, not a clinical decision tool.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv("data/healthcare_doctor_visits.csv")
print("Dataset shape:", df.shape)
display(df.head())
display(df.isna().sum().to_frame("Missing values"))
print("Duplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

## Doctor visit distribution

In [ ]:
counts = df["visits"].value_counts().sort_index()
plt.figure(figsize=(8,5))
plt.bar(counts.index.astype(str), counts.values)
plt.title("Distribution of Doctor Visits")
plt.xlabel("Number of visits"); plt.ylabel("Number of records")
plt.tight_layout(); plt.show()
print("Zero-visit percentage:", round(df["visits"].eq(0).mean()*100, 2))

## Group comparisons

In [ ]:
for col in ["gender", "private", "freepoor", "freerepat", "nchronic", "lchronic"]:
    print("\\n", col)
    display(df.groupby(col)["visits"].agg(["count", "mean", "median"]).round(3))

## Numerical correlations

In [ ]:
corr = df.select_dtypes(include=np.number).corr()["visits"].drop("visits").sort_values(ascending=False)
display(corr.to_frame("Correlation with visits"))
plt.figure(figsize=(8,5))
plt.barh(corr.sort_values().index, corr.sort_values().values)
plt.title("Correlation with Doctor Visits"); plt.xlabel("Pearson correlation")
plt.tight_layout(); plt.show()

## Exploratory machine-learning baseline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

X = df.drop(columns=["visits"])
y = df["visits"]
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
                          ("num", "passthrough", num_cols)])
model = Pipeline([("prep", prep), ("rf", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("MAE:", round(mean_absolute_error(y_test, pred), 3))
print("R2:", round(r2_score(y_test, pred), 3))

## Limitations
- Correlation does not establish causation.
- The visit count is concentrated at zero.
- The supplied guide does not define a conversion formula for the decimal age field, so it is not converted.
- This dataset and model are for educational analysis only, not medical diagnosis or clinical decision-making.